### PhayaThaiBERT(เพิ่ม) + Cross Validation(ต่อ)

In [2]:
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import gc
import seaborn as sns
import matplotlib.pyplot as plt
import os
import shutil

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,  
    DataCollatorWithPadding,
    Trainer
)

from datasets import Dataset, DatasetDict, load_from_disk

c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_name = "clicknext/phayathaibert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
full_ds = load_from_disk("../model")
full_ds = full_ds.rename_column("Label", "labels")
print(full_ds)

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 433
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 108
    })
})


In [5]:
labels = np.array(full_ds["train"]["labels"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
all_log_histories = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X=np.zeros(len(labels)), y=labels)):

    print(f"Fold {fold+1} / 5")

    train_fold = full_ds["train"].select(train_idx)
    val_fold   = full_ds["train"].select(val_idx)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,                   
        num_labels=2,                  
        ignore_mismatched_sizes=True  
    )

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        return_tensors="pt"
    )

    training_args = TrainingArguments(
        output_dir=f"./phayathaibert_fold_{fold+1}",
        num_train_epochs=4,            
        learning_rate=2e-5,         
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",         
        save_strategy="epoch",         
        load_best_model_at_end=True,     
        metric_for_best_model="eval_f1",  # เลือกตาม F1
        greater_is_better=True,
        logging_steps=10,
        seed=42,
        report_to="none",
        remove_unused_columns=False
        
    )

    def compute_metrics(pred):
        labels = pred.label_ids
        preds = np.argmax(pred.predictions, axis=-1)
        return {
        "accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average="binary")            
        }

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=val_fold,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

# Evaluate
    predictions = trainer.predict(val_fold)
    preds = np.argmax(predictions.predictions, axis=-1)
    true_labels = predictions.label_ids

    acc = accuracy_score(true_labels, preds)
    print(f"\n{'='*40}")
    print(f"Fold {fold+1} - Accuracy: {acc:.4f}")

    print("\nClassification Report:")
    print(classification_report(
            true_labels, preds,
            target_names=["Normal (0)", "Scam (1)"],
            zero_division=0))
                                
    fold_results.append({
    "fold": fold + 1,
    "accuracy": acc
    })

    mis_idx = np.where(preds != true_labels)[0]

    # Error Analysis
    mis_df = val_fold.select(mis_idx).to_pandas()
    mis_df['pred'] = preds[mis_idx]
    mis_df['true'] = true_labels[mis_idx]

    mis_df['prob'] = np.max(predictions.predictions[mis_idx], axis=1)
    mis_df[['true', 'pred', 'prob']].head(20)         

    #Fn and Fp
    fp_exa = (preds == 1) & (true_labels == 0)
    fn_exa = (preds == 0) & (true_labels == 1)

    print(f"False Positives: {fp_exa.sum()}")
    print(f"False Negatives: {fn_exa.sum()}")

                     
# ล้าง memory
    all_log_histories.append(trainer.state.log_history)
    del model, predictions, preds, true_labels, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



results_df = pd.DataFrame(fold_results)
print(results_df)
print(f"\nMean Accuracy: {results_df['accuracy'].mean():.2f} +/- {results_df['accuracy'].std():.2f}")


Fold 1 / 5


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2543.38it/s]
[transformers] CamembertForSequenceClassification LOAD REPORT from: clicknext/phayathaibert
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: Us

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.463033,0.306532,0.942529,0.938272
2,0.228008,0.172711,0.965517,0.963855
3,0.155040,0.159211,0.954023,0.952381
4,0.110929,0.153946,0.954023,0.952381


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.24s/it]
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.26s/it]
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.21s/it]
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing


Fold 1 - Accuracy: 0.9655

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.94      1.00      0.97        44
    Scam (1)       1.00      0.93      0.96        43

    accuracy                           0.97        87
   macro avg       0.97      0.97      0.97        87
weighted avg       0.97      0.97      0.97        87

False Positives: 0
False Negatives: 3
Fold 2 / 5


...


Fold 4 - Accuracy: 0.9186

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.97      0.86      0.92        44
    Scam (1)       0.87      0.98      0.92        42

    accuracy                           0.92        86
   macro avg       0.92      0.92      0.92        86
weighted avg       0.92      0.92      0.92        86

False Positives: 6
False Negatives: 1
Fold 5 / 5


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4046.59it/s]
[transformers] CamembertForSequenceClassification LOAD REPORT from: clicknext/phayathaibert
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: Us

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.448483,0.332720,0.883721,0.878049
2,0.190430,0.266061,0.872093,0.860759
3,0.152841,0.224078,0.918605,0.917647
4,0.130132,0.247156,0.883721,0.878049


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.29s/it]
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.68s/it]
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.21s/it]
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing


Fold 5 - Accuracy: 0.9186

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.91      0.93      0.92        43
    Scam (1)       0.93      0.91      0.92        43

    accuracy                           0.92        86
   macro avg       0.92      0.92      0.92        86
weighted avg       0.92      0.92      0.92        86

False Positives: 3
False Negatives: 4
   fold  accuracy
0     1  0.965517
1     2  0.908046
2     3  0.942529
3     4  0.918605
4     5  0.918605

Mean Accuracy: 0.93 +/- 0.02


In [4]:
import matplotlib.pyplot as plt

df = pd.DataFrame(all_log_histories[-1])

train = df[df['loss'].notna()]
eval_ = df[df['eval_loss'].notna()]

plt.figure(figsize=(8, 5))
plt.plot(train['epoch'], train['loss'], label='Train Loss', alpha=0.7, linestyle='--')
plt.plot(eval_['epoch'], eval_['eval_loss'], label='Eval Loss', marker='s', color='red', linewidth=2)

plt.title('Learning Curve (Check Overfit)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


NameError: name 'trainer' is not defined